In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schema names are loaded from 00_project_config.
# Reusable Spark transformation and validation helpers are loaded from
# 01_spark_utils.

BRONZE_TABLE = f"{BRONZE_SCHEMA}.api_competitions_data"
SILVER_TABLE = f"{SILVER_SCHEMA}.competitions_data"

BUSINESS_KEY = [
    "league_id",
]

REQUIRED_BRONZE_COLUMNS = [
    "response",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

REQUIRED_SILVER_COLUMNS = [
    "league_id",
    "league_name",
]


# -----------------------------------------------------------------------------
# Validate Bronze input
# -----------------------------------------------------------------------------
# Unlike the previous implementation, a missing Bronze table is treated as a
# pipeline failure rather than a warning. Producing an empty or stale Silver
# table would be more dangerous than failing visibly.

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )


# -----------------------------------------------------------------------------
# Read Bronze data
# -----------------------------------------------------------------------------
# The Bronze API response contains a top-level `response` array. Each element
# represents one competition and contains nested league and country data.
#
# Bronze remains source-oriented; Silver is responsible for flattening this
# structure into an analytical table.

bronze_df = spark.table(BRONZE_TABLE)

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Explode API response
# -----------------------------------------------------------------------------
# Each competition in the API response becomes one Silver candidate row.
#
# Technical Bronze metadata is retained so duplicate records created by
# repeated Bronze ingestion can be resolved using the latest ingestion event.

exploded_df = (
    bronze_df
    .select(
        F.explode("response").alias("response"),
        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),
        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),
        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Select and standardize Silver columns
# -----------------------------------------------------------------------------
# Source strings are trimmed and empty strings are converted to NULL using
# clean_string() from spark_utils.
#
# league_id remains numeric because API-Football provides it as a stable
# numeric source identifier.

silver_df = (
    exploded_df
    .select(
        F.col("response.league.id")
        .cast("long")
        .alias("league_id"),

        clean_string("response.league.name")
        .alias("league_name"),

        clean_string("response.league.type")
        .alias("league_type"),

        clean_string("response.country.code")
        .alias("country_code"),

        clean_string("response.country.name")
        .alias("country_name"),

        # Bronze lineage metadata
        F.col("_bronze_ingested_at"),
        F.col("_bronze_source_file"),
        F.col("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Future standardization placeholders
# -----------------------------------------------------------------------------
# The next project step will introduce canonical naming across sources.
#
# At that stage, fields such as league_name and country_name can be mapped to
# standardized platform values, while preserving the original source values.
#
# Example future pattern:
#
#   league_name_source
#   league_name_standardized
#
# or, preferably, standardized competition identifiers maintained through a
# dedicated mapping/conformed-dimension process.
#
# No placeholder columns are physically added yet because NULL-only columns
# would add noise without providing analytical value.


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# NULL business keys should be detected before deduplication. Otherwise
# multiple invalid rows with NULL league_id could be grouped together and one
# could be retained arbitrarily.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate competitions
# -----------------------------------------------------------------------------
# Bronze uses append mode, so the same raw API payload can appear multiple
# times across ingestion runs.
#
# For each league_id, keep the record from the latest Bronze ingestion.
#
# This is more deterministic and auditable than dropDuplicates(), which does
# not define which duplicate row should survive.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# league_id and league_name are required for a usable competition record.
#
# fail_on_null=True makes this an actual data-quality gate rather than only
# printing warnings and continuing with invalid Silver data.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.
# write_full_refresh() centralizes the Delta overwrite implementation.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"competition records into {SILVER_TABLE}."
)